# STR-packed R-tree: from bounding boxes to useful matches

This lab builds a small immutable STR-packed R-tree over geometry bounds. We will compare its envelope candidates with exact geometry intersections, then check the results against Shapely's production STRtree.

**You need:** Python, Shapely 2.1+, NumPy and Matplotlib.

The index stores bounding boxes. It can quickly rule out impossible matches, but a candidate may still fail the exact geometry predicate.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

repo_root = Path.cwd()
implementation_dir = repo_root / 'implementations/python/str-packed-rtree'
if not implementation_dir.exists():
    clone_dir = Path(tempfile.gettempdir()) / 'spatial-analysis'
    if not clone_dir.exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', 'https://github.com/geo-dev-hub/spatial-analysis.git', str(clone_dir)],
            check=True,
        )
    repo_root = clone_dir
    implementation_dir = repo_root / 'implementations/python/str-packed-rtree'
sys.path.insert(0, str(implementation_dir.resolve()))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from shapely import STRtree, box
from shapely.geometry import Polygon
from str_packed_rtree import STRPackedRTree


## 1. Create geometries with one deliberate false positive

Most examples are rectangles. One is an L-shaped polygon. Its bounding box covers an empty corner, so a query in that corner will be returned as an envelope candidate even though the polygon itself does not intersect the query.

In [ ]:
geometries = [
    box(0, 0, 1, 1),
    box(2, 0, 3, 1),
    Polygon([(4, 0), (5, 0), (5, 0.3), (4.3, 0.3), (4.3, 1), (4, 1), (4, 0)]),
    box(6, 0, 7, 1),
    box(0, 2, 1, 3),
    box(2, 2, 3, 3),
    box(4, 2, 5, 3),
    box(6, 2, 7, 3),
]

query = box(4.5, 0.6, 4.8, 0.9)
bounds = [geometry.bounds for geometry in geometries]
index = STRPackedRTree(bounds, node_capacity=3)

candidate_ids = index.query(query.bounds)
exact_ids = [i for i in candidate_ids if geometries[i].intersects(query)]

print('Envelope candidates:', candidate_ids)
print('Exact intersections:', exact_ids)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for i, geometry in enumerate(geometries):
    x, y = geometry.exterior.xy
    ax.fill(x, y, alpha=0.25)
    minx, miny, maxx, maxy = geometry.bounds
    ax.add_patch(Rectangle((minx, miny), maxx - minx, maxy - miny, fill=False, linestyle='--', linewidth=1))
    ax.text((minx + maxx) / 2, (miny + maxy) / 2, f'G{i}', ha='center', va='center')

qx, qy = query.exterior.xy
ax.fill(qx, qy, alpha=0.4, hatch='//')
ax.set_aspect('equal')
ax.set_title('Query overlaps the L-shape bounding box, not its geometry')
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.show()


## 2. Let Shapely refine candidates during the query

Shapely's STRtree can apply an exact predicate after its bounding-box search. The results should match the explicit candidate-then-refine sequence above.

In [ ]:
shapely_tree = STRtree(geometries)
shapely_candidates = shapely_tree.query(query)
shapely_exact = shapely_tree.query(query, predicate='intersects')

print('Shapely envelope candidates:', sorted(shapely_candidates.tolist()))
print('Shapely exact intersections:', sorted(shapely_exact.tolist()))
assert sorted(shapely_candidates.tolist()) == candidate_ids
assert sorted(shapely_exact.tolist()) == exact_ids


## 3. Query several windows at once

Shapely can query an array of input geometries. The result is a two-row array: the first row identifies the input query, and the second row identifies the matching tree geometry.

In [ ]:
queries = [box(0.2, 0.2, 0.8, 0.8), box(2.2, 2.2, 2.8, 2.8), query]
pair_indices = shapely_tree.query(queries, predicate='intersects')
print('Query index / tree index pairs:')
print(pair_indices.T.tolist())


## 4. Change node capacity

The node capacity changes how entries are grouped into the tree. It should not change the query results. The teaching implementation is checked against a brute-force scan in its test suite.

In [ ]:
for capacity in (2, 3, 4, 8, 10):
    candidate_tree = STRPackedRTree(bounds, node_capacity=capacity)
    print(f'capacity={capacity}: {candidate_tree.query(query.bounds)}')


## What to take away

- The R-tree organises bounding boxes hierarchically.
- STR packing sorts and tiles entries before grouping them into nodes.
- A query can prune branches whose bounding boxes do not intersect the query.
- Envelope candidates are not necessarily exact geometry matches.
- The Shapely STRtree is query-only and returns indices into the original input order.

Continue with the [STR-packed R-tree lesson](../algorithms/indexing/str-packed-rtree.md) and the [tested Python implementation](../implementations/python/str-packed-rtree/README.md).

**Reference:** [Shapely STRtree documentation](https://shapely.readthedocs.io/en/stable/strtree.html).